# Tahap 3 — Case Retrieval

Membangun vektor representasi dokumen dan model retrieval menggunakan dua pendekatan:
1. **TF-IDF** + SVM, Naive Bayes, Cosine Similarity
2. **IndoBERT** Embedding + Cosine Similarity

**Input**: `data/processed/cases.json`

**Output**:
- `data/eval/queries.json`
- `data/eval/retrieval_metrics.csv`
- `models/tfidf_vectorizer.pkl`, `models/svm_pasal.pkl`, `models/nb_pasal.pkl`
- `models/indobert_embeddings.npy`

## 3.1 Import Library

In [ ]:
import json
import pickle
import re
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, classification_report,
    f1_score, precision_score, recall_score,
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from transformers import AutoModel, AutoTokenizer

warnings.filterwarnings("ignore")

## 3.2 Konfigurasi dan Load Data

In [ ]:
DATA_PATH = Path("../data/processed/cases.json")
EVAL_DIR  = Path("../data/eval")
MODEL_DIR = Path("../models")

for d in [EVAL_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

with open(DATA_PATH, "r", encoding="utf-8") as f:
    cases = json.load(f)

print(f"Total kasus: {len(cases)}")

## 3.3 Preprocessing dan Splitting Data

In [ ]:
def preprocess_text(text):
    text = text.lower().strip()
    text = re.sub(r"disclaimer.*?ext\.318\)", "", text, flags=re.DOTALL)
    text = re.sub(r"halaman \d+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


for c in cases:
    c["text_processed"] = preprocess_text(c["text_full"])

labels_pasal = [c["label_pasal"] for c in cases]
train_cases, test_cases = train_test_split(
    cases, test_size=0.2, stratify=labels_pasal, random_state=42
)

print(f"Train: {len(train_cases)}, Test: {len(test_cases)}")
print(f"Train distribusi pasal: {dict(Counter([c['label_pasal'] for c in train_cases]))}")
print(f"Test distribusi pasal : {dict(Counter([c['label_pasal'] for c in test_cases]))}")

train_texts = [c["text_processed"] for c in train_cases]
test_texts = [c["text_processed"] for c in test_cases]
train_labels_pasal = [c["label_pasal"] for c in train_cases]
test_labels_pasal = [c["label_pasal"] for c in test_cases]

## 3.4 Fungsi Evaluasi

In [ ]:
def evaluate_model(y_true, y_pred, model_name, label_name):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    print(f"\nModel: {model_name} | Label: {label_name}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall   : {rec:.4f}")
    print(f"  F1-Score : {f1:.4f}")
    print(classification_report(y_true, y_pred, zero_division=0))
    return {
        "model": model_name, "label": label_name,
        "accuracy": round(acc, 4), "precision": round(prec, 4),
        "recall": round(rec, 4), "f1_score": round(f1, 4),
    }


def retrieval_predict(query_vectors, train_vectors, train_labels, k=5):
    predictions = []
    all_top_k = []
    for i in range(query_vectors.shape[0]):
        q = query_vectors[i:i+1]
        sims = cosine_similarity(q, train_vectors).flatten()
        top_k_idx = sims.argsort()[::-1][:k]
        top_k_labels = [train_labels[j] for j in top_k_idx]
        vote = Counter(top_k_labels).most_common(1)[0][0]
        predictions.append(vote)
        all_top_k.append(top_k_idx.tolist())
    return predictions, all_top_k


all_metrics = []

## 3.5 Pendekatan 1: TF-IDF + Machine Learning

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=5000, ngram_range=(1, 2),
    sublinear_tf=True, min_df=2, max_df=0.95,
)
X_train = vectorizer.fit_transform(train_texts)
X_test = vectorizer.transform(test_texts)
print(f"TF-IDF shape: train={X_train.shape}, test={X_test.shape}")

all_texts = [c["text_processed"] for c in cases]
all_tfidf_vectors = vectorizer.transform(all_texts)

with open(MODEL_DIR / "tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)
print("TF-IDF vectorizer disimpan")

### SVM Classification (label_pasal)

In [ ]:
svm_pasal = LinearSVC(max_iter=10000, random_state=42, C=1.0)
svm_pasal.fit(X_train, train_labels_pasal)
svm_pred = svm_pasal.predict(X_test)
m = evaluate_model(test_labels_pasal, svm_pred, "TF-IDF + SVM", "label_pasal")
all_metrics.append(m)

with open(MODEL_DIR / "svm_pasal.pkl", "wb") as f:
    pickle.dump(svm_pasal, f)

### Naive Bayes Classification (label_pasal)

In [ ]:
nb_pasal = MultinomialNB(alpha=1.0)
nb_pasal.fit(X_train, train_labels_pasal)
nb_pred = nb_pasal.predict(X_test)
m = evaluate_model(test_labels_pasal, nb_pred, "TF-IDF + NB", "label_pasal")
all_metrics.append(m)

with open(MODEL_DIR / "nb_pasal.pkl", "wb") as f:
    pickle.dump(nb_pasal, f)

### TF-IDF Cosine Retrieval (label_pasal, k=5)

In [ ]:
tfidf_ret_pred, _ = retrieval_predict(X_test, X_train, train_labels_pasal, k=5)
m = evaluate_model(test_labels_pasal, tfidf_ret_pred, "TF-IDF + Cosine (k=5)", "label_pasal")
all_metrics.append(m)

## 3.6 Pendekatan 2: IndoBERT Embedding

In [ ]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    summed = torch.sum(token_embeddings * mask_expanded, 1)
    clamped = torch.clamp(mask_expanded.sum(1), min=1e-9)
    return summed / clamped


def get_indobert_embeddings(texts, tokenizer, model, batch_size=4):
    device = get_device()
    model = model.to(device)
    model.eval()
    all_emb = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        batch_embs = []
        for text in batch:
            tokens = tokenizer(text, add_special_tokens=False, return_tensors="pt")["input_ids"][0]
            chunk_size = 510
            chunks = [tokens[j:j+chunk_size] for j in range(0, len(tokens), chunk_size)]
            chunk_embs = []
            for chunk in chunks:
                input_ids = torch.cat([
                    torch.tensor([tokenizer.cls_token_id]),
                    chunk,
                    torch.tensor([tokenizer.sep_token_id])
                ]).unsqueeze(0).to(device)
                attention_mask = torch.ones_like(input_ids).to(device)
                with torch.no_grad():
                    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                pooled = mean_pooling(outputs, attention_mask)
                chunk_embs.append(pooled)
            if chunk_embs:
                doc_emb = torch.stack(chunk_embs).mean(dim=0)
            else:
                doc_emb = torch.zeros((1, model.config.hidden_size)).to(device)
            batch_embs.append(doc_emb.cpu().numpy())
        all_emb.append(np.vstack(batch_embs))
        if (i // batch_size) % 5 == 0:
            print(f"  Embedding batch {i // batch_size + 1}/{(len(texts) + batch_size - 1) // batch_size}")
    return np.vstack(all_emb)


MODEL_NAME = "indobenchmark/indobert-base-p1"
device = get_device()
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
print("Model loaded")

In [ ]:
print("Generating train embeddings...")
train_emb = get_indobert_embeddings(train_texts, tokenizer, bert_model)
print(f"Train embeddings: {train_emb.shape}")

print("Generating test embeddings...")
test_emb = get_indobert_embeddings(test_texts, tokenizer, bert_model)
print(f"Test embeddings: {test_emb.shape}")

print("Generating all case embeddings...")
all_emb = get_indobert_embeddings(all_texts, tokenizer, bert_model)
np.save(MODEL_DIR / "indobert_embeddings.npy", all_emb)
print(f"All embeddings: {all_emb.shape}, disimpan")

### IndoBERT Cosine Retrieval (label_pasal, k=5)

In [ ]:
bert_ret_pred, _ = retrieval_predict(test_emb, train_emb, train_labels_pasal, k=5)
m = evaluate_model(test_labels_pasal, bert_ret_pred, "IndoBERT + Cosine (k=5)", "label_pasal")
all_metrics.append(m)

## 3.7 Fungsi retrieve() dan Generate queries.json

In [ ]:
tfidf_case_ids = [c["case_id"] for c in cases]
tfidf_case_labels = [c["label_pasal"] for c in cases]
bert_case_ids = tfidf_case_ids
bert_case_labels = tfidf_case_labels


def retrieve(query, k=5, method="tfidf"):
    query = preprocess_text(query)
    if method == "tfidf":
        q_vec = vectorizer.transform([query])
        sims = cosine_similarity(q_vec, all_tfidf_vectors).flatten()
        top_k_idx = sims.argsort()[::-1][:k]
        return [
            {"case_id": tfidf_case_ids[i], "label": tfidf_case_labels[i],
             "similarity": round(float(sims[i]), 4)}
            for i in top_k_idx
        ]
    elif method == "indobert":
        q_emb = get_indobert_embeddings([query], tokenizer, bert_model)
        sims = cosine_similarity(q_emb, all_emb).flatten()
        top_k_idx = sims.argsort()[::-1][:k]
        return [
            {"case_id": bert_case_ids[i], "label": bert_case_labels[i],
             "similarity": round(float(sims[i]), 4)}
            for i in top_k_idx
        ]


queries = []
for i, tc in enumerate(test_cases[:10]):
    tfidf_results = retrieve(tc["text_processed"], k=5, method="tfidf")
    bert_results = retrieve(tc["text_processed"], k=5, method="indobert")
    q = {
        "query_id": f"q{i+1:02d}",
        "source_case_id": tc["case_id"],
        "ground_truth_label_pasal": tc["label_pasal"],
        "ground_truth_label_vonis": tc["label_vonis"],
        "query_text": tc["text_processed"][:500],
        "tfidf_top5": tfidf_results,
        "indobert_top5": bert_results,
    }
    queries.append(q)
    print(f"  {q['query_id']}: {tc['case_id']} ({tc['label_pasal']}) -> "
          f"TF-IDF top1={tfidf_results[0]['case_id']} | BERT top1={bert_results[0]['case_id']}")

with open(EVAL_DIR / "queries.json", "w", encoding="utf-8") as f:
    json.dump(queries, f, ensure_ascii=False, indent=2)
print(f"\nqueries.json disimpan: {len(queries)} queries")

## 3.8 Simpan Retrieval Metrics

In [ ]:
metrics_df = pd.DataFrame(all_metrics)
metrics_df.to_csv(EVAL_DIR / "retrieval_metrics.csv", index=False)
print("RETRIEVAL METRICS SUMMARY")
metrics_df

## 3.9 Demo retrieve()

In [ ]:
demo_query = test_cases[0]["text_processed"][:300]
print(f"Query (300 char): {demo_query[:100]}...")
print(f"Ground truth: {test_cases[0]['label_pasal']}")

print("\n--- TF-IDF Retrieval ---")
for r in retrieve(demo_query, k=5, method="tfidf"):
    print(f"  {r['case_id']} | label={r['label']} | sim={r['similarity']}")

print("\n--- IndoBERT Retrieval ---")
for r in retrieve(demo_query, k=5, method="indobert"):
    print(f"  {r['case_id']} | label={r['label']} | sim={r['similarity']}")